# Customer Churn Prediction — Exploratory Data Analysis & Cleaning

This notebook explores the raw telecom customer dataset, documents data-quality
issues, cleans them, and visualizes the key drivers of churn before modeling.

**Sections**
1. Load & inspect raw data
2. Data-quality audit (missingness, duplicates, inconsistent text)
3. Cleaning
4. Feature engineering
5. Exploratory visualizations
6. Key findings

In [ ]:
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from data_preprocessing import clean_data, engineer_features

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', None)

## 1. Load & inspect raw data

In [ ]:
raw = pd.read_csv('../data/raw/customer_churn.csv')
print(raw.shape)
raw.head()

In [ ]:
raw.info()

In [ ]:
raw.describe(include='all').T

## 2. Data-quality audit

Before modeling, let's find the issues you'd expect in a real production export:
missing values, duplicate rows, and inconsistent category text.

In [ ]:
print('Missing values per column:')
print(raw.isna().sum()[raw.isna().sum() > 0])
print()
print(f"TotalCharges dtype: {raw['TotalCharges'].dtype}")
print(f"Rows where TotalCharges is not purely numeric: "
      f"{pd.to_numeric(raw['TotalCharges'], errors='coerce').isna().sum()}")

In [ ]:
print(f"Exact duplicate rows: {raw.duplicated().sum()}")
print(f"Unique gender values (raw): {raw['gender'].unique()}")

## 3. Cleaning

`clean_data()` (from `src/data_preprocessing.py`) handles:
- dropping exact duplicates
- trimming/normalizing whitespace & casing in categorical columns
- coercing `TotalCharges` to numeric and imputing missing values as `tenure * MonthlyCharges`

In [ ]:
clean = clean_data(raw)
print('Before:', raw.shape, '-> After:', clean.shape)
print('Remaining missing values:', clean.isna().sum().sum())
print('Unique gender values (clean):', clean['gender'].unique())

## 4. Feature engineering

`engineer_features()` adds:
- `AvgMonthlySpend` — TotalCharges / tenure (falls back to MonthlyCharges for brand-new customers)
- `TenureGroup` — bucketed tenure (0-1yr / 1-2yr / 2-4yr / 4+yr)
- `NumServices` — count of add-on services subscribed to

In [ ]:
df = engineer_features(clean)
df[['tenure', 'AvgMonthlySpend', 'TenureGroup', 'NumServices']].head()

## 5. Exploratory visualizations

In [ ]:
churn_rate = (df['Churn'] == 'Yes').mean()
print(f'Overall churn rate: {churn_rate:.1%}')

fig, ax = plt.subplots(figsize=(5, 4))
df['Churn'].value_counts().plot(kind='bar', color=['#4C72B0', '#DD8452'], ax=ax)
ax.set_title('Churn Distribution')
ax.set_xlabel('Churn')
ax.set_ylabel('Customers')
plt.xticks(rotation=0)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
rate_by_contract = df.groupby('Contract')['Churn'].apply(lambda s: (s == 'Yes').mean()).sort_values(ascending=False)
rate_by_contract.plot(kind='bar', color='#C44E52', ax=ax)
ax.set_title('Churn Rate by Contract Type')
ax.set_ylabel('Churn Rate')
plt.xticks(rotation=20)
plt.show()
rate_by_contract

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.boxplot(data=df, x='Churn', y='MonthlyCharges', ax=ax, hue='Churn', legend=False,
            palette={'Yes': '#DD8452', 'No': '#4C72B0'})
ax.set_title('Monthly Charges by Churn Status')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
rate_by_tenure = df.groupby('TenureGroup')['Churn'].apply(lambda s: (s == 'Yes').mean())
rate_by_tenure = rate_by_tenure.reindex(['0-1 yr', '1-2 yr', '2-4 yr', '4+ yr'])
rate_by_tenure.plot(kind='bar', color='#55A868', ax=ax)
ax.set_title('Churn Rate by Tenure Group')
ax.set_ylabel('Churn Rate')
plt.xticks(rotation=0)
plt.show()
rate_by_tenure

In [ ]:
numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'AvgMonthlySpend', 'NumServices']
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(df[numeric_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm', ax=ax)
ax.set_title('Correlation Between Numeric Features')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
rate_by_internet = df.groupby('InternetService')['Churn'].apply(lambda s: (s == 'Yes').mean()).sort_values(ascending=False)
rate_by_internet.plot(kind='bar', color='#8172B2', ax=ax)
ax.set_title('Churn Rate by Internet Service Type')
ax.set_ylabel('Churn Rate')
plt.xticks(rotation=0)
plt.show()
rate_by_internet

## 6. Key findings

- **Month-to-month contracts churn far more** than one- or two-year contracts — this is
  consistently the single strongest churn driver in this dataset.
- **New customers are highest risk.** Churn rate drops sharply as tenure increases past
  the first year, then continues declining.
- **Fiber optic internet customers churn more** than DSL or no-internet customers, despite
  (or because of) paying the highest monthly bills.
- **Add-on services correlate with retention** — customers without Online Security or Tech
  Support churn noticeably more than those with them.
- These patterns directly motivate the features used in `src/train_model.py`, and match
  what the trained model later ranks as top predictive features (see
  `reports/figures/feature_importance.png`).

Next: open `../src/train_model.py` (or just run `python src/train_model.py` from the repo
root) to train and evaluate the classification models.